# RAG Primer (Fast) — Retrieval-Augmented Generation



**Goal:** understand how RAG lets a system answer from an external knowledge base instead of guessing.

**Why this version is fast:** the core runs in *seconds* with **no large model downloads** — retrieval uses TF-IDF from scikit-learn (already installed). A clearly-marked **optional** section at the end plugs in a real neural embedding model + LLM if you want to see that too (one-time ~300 MB download).

### The 5 stages of RAG
1. **Embed / Vectorize** — turn each document into a vector.
2. **Index** — store the vectors so we can search them.
3. **Retrieve** — find the documents most similar to the question.
4. **Augment** — put the retrieved text into the prompt as context.
5. **Generate** — produce an answer grounded in that context.

In [1]:
# A tiny knowledge base: the facts our system is allowed to look things up in.
knowledge_base = [
    "The Eiffel Tower is located in Paris, France.",
    "The Great Wall of China stretches thousands of kilometres across northern China.",
    "Python is a popular programming language for AI and machine learning.",
    "The human brain contains around 86 billion neurons.",
    "Photosynthesis in plants converts sunlight into chemical energy.",
    "Mount Everest is the tallest mountain above sea level, in the Himalayas.",
]

for i, doc in enumerate(knowledge_base):
    print(f"[{i}] {doc}")

[0] The Eiffel Tower is located in Paris, France.
[1] The Great Wall of China stretches thousands of kilometres across northern China.
[2] Python is a popular programming language for AI and machine learning.
[3] The human brain contains around 86 billion neurons.
[4] Photosynthesis in plants converts sunlight into chemical energy.
[5] Mount Everest is the tallest mountain above sea level, in the Himalayas.


## Stages 1 & 2 — Vectorize & Index (TF-IDF)

A computer can't compare sentences as raw text, so we turn each one into a **vector of numbers**. Here we use **TF-IDF**, which scores each word by how important it is to a document. Sentences that share important words get similar vectors. This needs **no download** and is instant.

> In production RAG you'd usually swap TF-IDF for **neural embeddings** (like the word embeddings from Module 2, or a model such as MiniLM). We show that in the optional section at the end.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fit the vectorizer on our knowledge base and store the document vectors (our "index").
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(knowledge_base)

print("Indexed", doc_vectors.shape[0], "documents,",
      "each represented as a", doc_vectors.shape[1], "-dimensional vector.")

Indexed 6 documents, each represented as a 38 -dimensional vector.


## Stage 3 — Retrieve

We vectorize the **question** the same way, then rank documents by **cosine similarity** (1.0 = same direction, 0.0 = nothing in common). The highest score is the most relevant document.


In [3]:
def retrieve(query, top_k=2):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, doc_vectors)[0]
    ranked = scores.argsort()[::-1][:top_k]   # indices of highest scores first
    return [(knowledge_base[i], float(scores[i])) for i in ranked]

# Notice the most relevant document has the highest score.
query = "Where is the Eiffel Tower?"
print("Query:", query)
for doc, score in retrieve(query):
    print(f"  score={score:.3f}  ->  {doc}")

Query: Where is the Eiffel Tower?
  score=0.632  ->  The Eiffel Tower is located in Paris, France.
  score=0.000  ->  Mount Everest is the tallest mountain above sea level, in the Himalayas.


## Stage 4 — Augment

"Augment" simply means we build a prompt that **includes the retrieved documents as context**. Printing the prompt lets you see exactly what an LLM would receive.


In [4]:
def build_prompt(query, retrieved):
    context = "\n".join(doc for doc, _score in retrieved)
    return (
        "Answer the question using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\nAnswer:"
    )

print(build_prompt("Where is the Eiffel Tower?", retrieve("Where is the Eiffel Tower?")))

Answer the question using only the context below.

Context:
The Eiffel Tower is located in Paris, France.
Mount Everest is the tallest mountain above sea level, in the Himalayas.

Question: Where is the Eiffel Tower?
Answer:


## Stage 5 — Generate (fast, no LLM)

So the notebook stays instant, our default "generator" is a tiny stand-in: it returns the best-matching document as the answer. Crucially, if **nothing** in the knowledge base is relevant (top score ≈ 0), it says so instead of making something up — that honesty is one of RAG's biggest benefits.

*(The optional section below replaces this stand-in with a real LLM.)*


In [5]:
def rag_answer(query, top_k=2):
    retrieved = retrieve(query, top_k=top_k)
    best_doc, best_score = retrieved[0]
    if best_score < 0.05:   # nothing relevant found
        return "I don't have information about that in my knowledge base.", retrieved
    return best_doc, retrieved

# In-knowledge questions -> grounded answers
for q in ["What city is the Eiffel Tower in?", "How many neurons are in the brain?"]:
    answer, _ = rag_answer(q)
    print(f"Q: {q}\nA: {answer}\n")

# Out-of-knowledge question -> RAG declines instead of hallucinating
q = "Who won the 2022 football World Cup?"
answer, retrieved = rag_answer(q)
print(f"Q: {q}\nA: {answer}")
print(f"(best retrieval score was only {retrieved[0][1]:.3f})")

Q: What city is the Eiffel Tower in?
A: The Eiffel Tower is located in Paris, France.

Q: How many neurons are in the brain?
A: The human brain contains around 86 billion neurons.

Q: Who won the 2022 football World Cup?
A: I don't have information about that in my knowledge base.
(best retrieval score was only 0.000)


**Takeaway:** the system can only answer what it can **retrieve**. Give it the right document and it answers correctly; ask something outside its knowledge base and it declines instead of guessing. That grounding is the whole point of RAG.


In [6]:
# 👉 Try it yourself: change the question and re-run (instant, no downloads).
student_query = "What does photosynthesis do?"
answer, retrieved = rag_answer(student_query)
print("Question:", student_query)
print("Answer:", answer)
print("Sources:", [doc for doc, _ in retrieved])

Question: What does photosynthesis do?
Answer: Photosynthesis in plants converts sunlight into chemical energy.
Sources: ['Photosynthesis in plants converts sunlight into chemical energy.', 'Mount Everest is the tallest mountain above sea level, in the Himalayas.']


## Optional — real neural embeddings + a real LLM

Everything above ran instantly with no downloads. This **optional** section shows "full" RAG with:
- **Neural embeddings** (MiniLM) instead of TF-IDF, and
- **A real generator** (Flan-T5-small) instead of returning a sentence.

⚠️ Running the next cell downloads models **once** (~300 MB total, then cached). Skip it if you're short on time — you already understand RAG from the stages above.


In [7]:
# OPTIONAL (one-time ~300 MB download). Real retrieve -> augment -> generate with an LLM.
# Flan-T5 is a small seq2seq model: it reads a prompt and writes an answer.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def llm_rag(query, top_k=2):
    retrieved = retrieve(query, top_k=top_k)          # reuse the TF-IDF retriever above
    prompt = build_prompt(query, retrieved)
    input_ids = tokenizer(prompt, return_tensors="pt", truncation=True).input_ids
    output_ids = llm.generate(input_ids, max_new_tokens=60)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True), retrieved

question = "What city is the Eiffel Tower in?"

# Without retrieval, the model answers from memory only (often wrong for niche facts):
no_ctx = tokenizer(f"Question: {question}\nAnswer:", return_tensors="pt").input_ids
print("WITHOUT context:", tokenizer.decode(llm.generate(no_ctx, max_new_tokens=20)[0], skip_special_tokens=True))

# With retrieval, the answer is grounded in the knowledge base:
answer, _ = llm_rag(question)
print("WITH context (RAG):", answer)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


WITHOUT context: santo juan
WITH context (RAG): Paris


## Reflect & next steps
1. Why does the system decline the World Cup question? How does that reduce hallucination?
2. What did the retrieval scores tell you about which document was used?
3. In the optional section, how did the LLM's answer change *with* vs *without* context?

**Next:** in `agentic_rag.ipynb` the agent *decides for itself* when to retrieve — turning this fixed pipeline into an agent.
